In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision timm scikit-learn

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import timm
import numpy as np
import math

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [ ]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(8),
    T.ColorJitter(0.15, 0.15, 0.15),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dir = "/content/drive/MyDrive/RAF-DB/DATASET/train"
test_dir  = "/content/drive/MyDrive/RAF-DB/DATASET/test"

train_ds = ImageFolder(train_dir, transform=train_transform)
test_ds  = ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
class_counts = np.bincount(train_ds.targets)
class_weights = 1. / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = torch.FloatTensor(class_weights).to(device)

In [ ]:
model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True,
    num_classes=7
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=1e-4
)

In [ ]:
epochs = 80
warmup_epochs = 5

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return float(epoch) / float(max(1, warmup_epochs))
    return 0.5 * (1 + math.cos(
        math.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)
    ))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
scaler = torch.amp.GradScaler('cuda')
best_acc = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(x)
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()

        # Gradient clipping (Transformer stability)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    scheduler.step()

    # ---- Validation after each epoch ----
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            preds = model(x).argmax(1)
            y_true.extend(y.numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_swin_rafdb.pth")

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Val Acc: {acc:.4f}")

Epoch [1/80] Loss: 2.1833 Val Acc: 0.0786
Epoch [2/80] Loss: 2.0001 Val Acc: 0.4374
Epoch [3/80] Loss: 1.6850 Val Acc: 0.6017
Epoch [4/80] Loss: 1.5045 Val Acc: 0.6750
Epoch [5/80] Loss: 1.3848 Val Acc: 0.7601
Epoch [6/80] Loss: 1.2959 Val Acc: 0.7174
Epoch [7/80] Loss: 1.1943 Val Acc: 0.7842
Epoch [8/80] Loss: 1.1318 Val Acc: 0.8054
Epoch [9/80] Loss: 1.0730 Val Acc: 0.8064
Epoch [10/80] Loss: 1.0200 Val Acc: 0.8286
Epoch [11/80] Loss: 0.9795 Val Acc: 0.7979
Epoch [12/80] Loss: 0.9472 Val Acc: 0.8253
Epoch [13/80] Loss: 0.9199 Val Acc: 0.8243
Epoch [14/80] Loss: 0.9009 Val Acc: 0.8334
Epoch [15/80] Loss: 0.8889 Val Acc: 0.8468
Epoch [16/80] Loss: 0.8684 Val Acc: 0.8504
Epoch [17/80] Loss: 0.8538 Val Acc: 0.8442
Epoch [18/80] Loss: 0.8497 Val Acc: 0.8087
Epoch [19/80] Loss: 0.8341 Val Acc: 0.8484
Epoch [20/80] Loss: 0.8335 Val Acc: 0.8530
Epoch [21/80] Loss: 0.8261 Val Acc: 0.8458
Epoch [22/80] Loss: 0.8185 Val Acc: 0.8550
Epoch [23/80] Loss: 0.8151 Val Acc: 0.8220
Epoch [24/80] Loss: 

In [ ]:
model.load_state_dict(torch.load("best_swin_rafdb.pth"))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x).argmax(1)
        y_true.extend(y.numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
f1        = f1_score(y_true, y_pred, average='weighted')
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

Accuracy : 0.8768
Precision: 0.876
Recall   : 0.8768
F1 Score : 0.876


In [ ]:
import os

base_path = "/content/drive/MyDrive/facial_emotion_project"
models_path = os.path.join(base_path, "models")

os.makedirs(models_path, exist_ok=True)

print("Folders created:", os.path.exists(models_path))

Folders created: True


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/facial_emotion_project/models/SwinFaceViT.pth"
)

In [ ]:
import os
os.path.isfile("/content/drive/MyDrive/facial_emotion_project/models/SwinFaceViT.pth")

True